# 03 — Формування об'єднаного набору даних ВВП та нічного освітлення

У цьому ноутбуці виконується інтеграція річних показників нічного освітлення (NTL), отриманих на основі даних VIIRS VNP46A4, із річними даними ВВП Eurostat.

Результатом є панельний набір даних для 36 європейських країн за 2012–2025 роки, який використовується на наступному етапі для побудови моделей.

## 1. Налаштування параметрів та задання шляхів до даних

In [ ]:
from google.colab import drive
from pathlib import Path

import pandas as pd

# Підключення Google Drive
drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GDP_NTL_Project"
)

GDP_PATH = (
    PROJECT_ROOT
    / "data"
    / "economic"
    / "eurostat_nama_10_gdp_2012_2025.csv"
)

NTL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ntl"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

START_YEAR = 2012
END_YEAR = 2025

print("Project exists:", PROJECT_ROOT.exists())
print("GDP exists:", GDP_PATH.exists())
print("NTL directory exists:", NTL_DIR.exists())

if NTL_DIR.exists():
    print("\nNTL files:")
    for path in sorted(NTL_DIR.glob("*.csv")):
        print(path.name)

Mounted at /content/drive
Project exists: True
GDP exists: True
NTL directory exists: True

NTL files:
vnp46a4_country_ntl_2012.csv
vnp46a4_country_ntl_2013.csv
vnp46a4_country_ntl_2014.csv
vnp46a4_country_ntl_2015.csv
vnp46a4_country_ntl_2016.csv
vnp46a4_country_ntl_2017.csv
vnp46a4_country_ntl_2018.csv
vnp46a4_country_ntl_2019.csv
vnp46a4_country_ntl_2020.csv
vnp46a4_country_ntl_2021.csv
vnp46a4_country_ntl_2022.csv
vnp46a4_country_ntl_2023.csv
vnp46a4_country_ntl_2024.csv
vnp46a4_country_ntl_2025.csv
vnp46a4_processing_summary_2012_2025.csv


## 2. Завантаження та перевірка даних нічного освітлення

In [ ]:
ntl_files = [
    NTL_DIR / f"vnp46a4_country_ntl_{year}.csv"
    for year in range(START_YEAR, END_YEAR + 1)
]

missing_files = [
    path.name
    for path in ntl_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing NTL files: {missing_files}"
    )

ntl = pd.concat(
    [
        pd.read_csv(path)
        for path in ntl_files
    ],
    ignore_index=True
)

ntl = (
    ntl
    .sort_values(["CNTR_ID", "year"])
    .reset_index(drop=True)
)

print("Observations:", len(ntl))
print("Countries:", ntl["CNTR_ID"].nunique())
print(
    "Years:",
    ntl["year"].min(),
    "—",
    ntl["year"].max()
)
print(
    "Duplicate country-years:",
    ntl.duplicated(
        ["CNTR_ID", "year"]
    ).sum()
)

ntl.head()


Observations: 504
Countries: 36
Years: 2012 — 2025
Duplicate country-years: 0


,CNTR_ID,NAME_ENGL,year,country_area_km2,n_valid_pixels,valid_ntl_area_km2,ntl_mean,ntl_median,ntl_std,ntl_p25,...,invalid_radiance_km2,water_share,qa0_share,qa1_share,qa2_share,invalid_radiance_share,valid_ntl_area_km2_qa01,ntl_mean_area_weighted_qa01,ntl_sum_area_weighted_qa01,ntl_coverage_share_qa01
0,AL,Albania,2012,28806.031029,175875,28468.848979,0.478369,0.0,3.670705,0.0,...,0.0,0.011514,0.988295,0.000163,0.000000,0.0,28473.541699,0.478058,13612.009732,0.988458
1,AL,Albania,2013,28806.031029,175872,28468.358826,0.473239,0.0,3.716385,0.0,...,0.0,0.011536,0.988278,0.000158,0.000000,0.0,28472.899331,0.472847,13463.311158,0.988435
2,AL,Albania,2014,28806.031029,175869,28467.871955,0.448598,0.0,3.542073,0.0,...,0.0,0.011598,0.988261,0.000113,0.000000,0.0,28471.125460,0.448293,12763.419815,0.988374
3,AL,Albania,2015,28806.031029,175879,28469.475546,0.423677,0.0,3.616563,0.0,...,0.0,0.011553,0.988316,0.000102,0.000000,0.0,28472.418578,0.423596,12060.788738,0.988419
4,AL,Albania,2016,28806.031029,175891,28471.414757,0.406204,0.0,3.802593,0.0,...,0.0,0.011474,0.988384,0.000108,0.000006,0.0,28474.518672,0.406305,11569.331486,0.988492


In [ ]:
NTL_FEATURES = [
    "ntl_sum_area_weighted",
    "ntl_mean",
    "ntl_mean_area_weighted",
    "ntl_median",
    "ntl_std",
    "ntl_p25",
    "ntl_p75",
    "ntl_p90",
    "ntl_p95",
    "ntl_iqr",
    "ntl_cv",
    "lit_area_km2",
    "lit_area_share_valid",
    "ntl_positive_mean",
]

ntl_check = pd.DataFrame({
    "feature": NTL_FEATURES,
    "missing": [
        ntl[col].isna().sum()
        for col in NTL_FEATURES
    ]
})

display(ntl_check)

print(
    "QA0 coverage:",
    round(ntl["ntl_coverage_share"].min(), 4),
    "—",
    round(ntl["ntl_coverage_share"].max(), 4)
)

print(
    "QA01 coverage:",
    round(ntl["ntl_coverage_share_qa01"].min(), 4),
    "—",
    round(ntl["ntl_coverage_share_qa01"].max(), 4)
)

,feature,missing
0,ntl_sum_area_weighted,0
1,ntl_mean,0
2,ntl_mean_area_weighted,0
3,ntl_median,0
4,ntl_std,0
5,ntl_p25,0
6,ntl_p75,0
7,ntl_p90,0
8,ntl_p95,0
9,ntl_iqr,0


QA0 coverage: 0.6668 — 1.0047
QA01 coverage: 0.9461 — 1.0047


## 3. Завантаження та перетворення даних ВВП

In [ ]:
gdp_raw = pd.read_csv(GDP_PATH)

print("Shape:", gdp_raw.shape)
print("Columns:")
print(gdp_raw.columns.tolist())

gdp_raw.head()

Shape: (36, 18)
Columns:
['freq', 'unit', 'na_item', 'geo\\TIME_PERIOD', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


,freq,unit,na_item,geo\TIME_PERIOD,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,CP_MEUR,B1GQ,AL,9578.6,9669.8,10018.0,10371.5,10922.7,11760.0,13019.2,13881.7,13430.2,15272.1,18075.0,21738.8,24995.3,27075.3
1,A,CP_MEUR,B1GQ,AT,316589.4,321191.7,330113.5,342083.5,355665.6,367294.9,383234.3,395706.8,380317.9,406231.5,449382.2,477837.3,494087.6,514328.1
2,A,CP_MEUR,B1GQ,BA,13407.5,13691.8,13988.3,14791.1,15474.3,16260.5,17354.2,18296.5,17755.9,20014.7,23324.1,25523.8,27483.1,28913.9
3,A,CP_MEUR,B1GQ,BE,387934.9,394616.3,404958.3,415538.0,428467.1,443407.2,459491.8,479444.9,463750.9,506047.2,561678.2,601866.5,619914.3,642015.3
4,A,CP_MEUR,B1GQ,BG,42254.6,42055.5,43024.0,45797.1,48751.4,52501.0,55999.7,61194.4,61855.6,71343.9,86078.2,94525.1,104767.2,116018.3


In [ ]:
year_cols = [
    str(year)
    for year in range(START_YEAR, END_YEAR + 1)
]

gdp = gdp_raw.melt(
    id_vars=[
        "freq",
        "unit",
        "na_item",
        "geo\\TIME_PERIOD"
    ],
    value_vars=year_cols,
    var_name="year",
    value_name="gdp_meur"
)

gdp = gdp.rename(
    columns={
        "geo\\TIME_PERIOD": "CNTR_ID"
    }
)

gdp["year"] = gdp["year"].astype(int)
gdp["gdp_meur"] = pd.to_numeric(
    gdp["gdp_meur"],
    errors="coerce"
)

gdp = (
    gdp[
        [
            "CNTR_ID",
            "year",
            "gdp_meur"
        ]
    ]
    .sort_values(["CNTR_ID", "year"])
    .reset_index(drop=True)
)

print("Observations:", len(gdp))
print("Countries:", gdp["CNTR_ID"].nunique())
print(
    "Years:",
    gdp["year"].min(),
    "—",
    gdp["year"].max()
)
print(
    "Duplicate country-years:",
    gdp.duplicated(
        ["CNTR_ID", "year"]
    ).sum()
)
print(
    "Missing GDP:",
    gdp["gdp_meur"].isna().sum()
)

display(gdp.head(20))

Observations: 504
Countries: 36
Years: 2012 — 2025
Duplicate country-years: 0
Missing GDP: 2


,CNTR_ID,year,gdp_meur
0,AL,2012,9578.6
1,AL,2013,9669.8
2,AL,2014,10018.0
3,AL,2015,10371.5
4,AL,2016,10922.7
5,AL,2017,11760.0
6,AL,2018,13019.2
7,AL,2019,13881.7
8,AL,2020,13430.2
9,AL,2021,15272.1


## 4. Об'єднання даних ВВП та нічного освітлення

In [ ]:
dataset = ntl.merge(
    gdp,
    on=["CNTR_ID", "year"],
    how="left",
    validate="one_to_one"
)

dataset = (
    dataset
    .sort_values(["CNTR_ID", "year"])
    .reset_index(drop=True)
)

print("Observations:", len(dataset))
print("Countries:", dataset["CNTR_ID"].nunique())

print(
    "Duplicate country-years:",
    dataset.duplicated(
        ["CNTR_ID", "year"]
    ).sum()
)

print(
    "Missing GDP:",
    dataset["gdp_meur"].isna().sum()
)

print(
    "Missing QA0 NTL:",
    dataset[
        "ntl_mean_area_weighted"
    ].isna().sum()
)

print(
    "Missing QA01 NTL:",
    dataset[
        "ntl_mean_area_weighted_qa01"
    ].isna().sum()
)

display(
    dataset.loc[
        dataset["gdp_meur"].isna(),
        [
            "CNTR_ID",
            "NAME_ENGL",
            "year",
            "gdp_meur"
        ]
    ]
)

Observations: 504
Countries: 36
Duplicate country-years: 0
Missing GDP: 2
Missing QA0 NTL: 0
Missing QA01 NTL: 0


,CNTR_ID,NAME_ENGL,year,gdp_meur
280,LI,Liechtenstein,2012,NaN
293,LI,Liechtenstein,2025,NaN


## 5. Фінальна перевірка та збереження панельного датасету

In [ ]:
OUTPUT_PATH = (
    OUTPUT_DIR
    / "gdp_ntl_panel_2012_2025.csv"
)

expected_years = END_YEAR - START_YEAR + 1
expected_countries = ntl["CNTR_ID"].nunique()
expected_observations = (
    expected_countries * expected_years
)

# Перевірка структури панелі
assert len(dataset) == expected_observations
assert dataset["CNTR_ID"].nunique() == expected_countries
assert dataset["year"].nunique() == expected_years

assert (
    dataset.duplicated(
        ["CNTR_ID", "year"]
    ).sum()
    == 0
)

# Перевірка основних NTL-ознак
assert dataset[NTL_FEATURES].notna().all().all()

# Перевірка альтернативної QA01-специфікації
assert (
    dataset["ntl_mean_area_weighted_qa01"]
    .notna()
    .all()
)

assert (
    dataset["ntl_sum_area_weighted_qa01"]
    .notna()
    .all()
)

print(
    "Кількість пропущених значень ВВП:",
    dataset["gdp_meur"].isna().sum()
)

print(
    "Кількість пропусків у 14 основних NTL-ознаках:",
    dataset[NTL_FEATURES].isna().sum().sum()
)

# Збереження
dataset.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Датасет успішно збережено.")
print("Розмір:", dataset.shape)
print("Шлях:", OUTPUT_PATH)

display(dataset)

Кількість пропущених значень ВВП: 2
Кількість пропусків у 14 основних NTL-ознаках: 0
Датасет успішно збережено.
Розмір: (504, 37)
Шлях: /content/drive/MyDrive/GDP_NTL_Project/data/processed/gdp_ntl_panel_2012_2025.csv


,CNTR_ID,NAME_ENGL,year,country_area_km2,n_valid_pixels,valid_ntl_area_km2,ntl_mean,ntl_median,ntl_std,ntl_p25,...,water_share,qa0_share,qa1_share,qa2_share,invalid_radiance_share,valid_ntl_area_km2_qa01,ntl_mean_area_weighted_qa01,ntl_sum_area_weighted_qa01,ntl_coverage_share_qa01,gdp_meur
0,AL,Albania,2012,28806.031029,175875,28468.848979,0.478369,0.0,3.670705,0.0,...,0.011514,0.988295,0.000163,0.000000,0.0,28473.541699,0.478058,13612.009732,0.988458,9578.6
1,AL,Albania,2013,28806.031029,175872,28468.358826,0.473239,0.0,3.716385,0.0,...,0.011536,0.988278,0.000158,0.000000,0.0,28472.899331,0.472847,13463.311158,0.988435,9669.8
2,AL,Albania,2014,28806.031029,175869,28467.871955,0.448598,0.0,3.542073,0.0,...,0.011598,0.988261,0.000113,0.000000,0.0,28471.125460,0.448293,12763.419815,0.988374,10018.0
3,AL,Albania,2015,28806.031029,175879,28469.475546,0.423677,0.0,3.616563,0.0,...,0.011553,0.988316,0.000102,0.000000,0.0,28472.418578,0.423596,12060.788738,0.988419,10371.5
4,AL,Albania,2016,28806.031029,175891,28471.414757,0.406204,0.0,3.802593,0.0,...,0.011474,0.988384,0.000108,0.000006,0.0,28474.518672,0.406305,11569.331486,0.988492,10922.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,SK,Slovakia,2021,49015.978686,344501,48949.813062,0.683477,0.0,3.604036,0.0,...,0.001070,0.998650,0.000255,0.000000,0.0,48962.315234,0.685631,33570.091675,0.998905,101891.6
500,SK,Slovakia,2022,49015.978686,344527,48953.475710,0.740721,0.0,3.656423,0.0,...,0.001062,0.998725,0.000189,0.000000,0.0,48962.739374,0.743053,36381.920696,0.998914,109959.8
501,SK,Slovakia,2023,49015.978686,344531,48954.085292,0.704482,0.0,3.336240,0.0,...,0.001056,0.998737,0.000182,0.000000,0.0,48963.025754,0.706696,34601.953162,0.998920,123538.7
502,SK,Slovakia,2024,49015.978686,344486,48947.627945,0.722511,0.0,3.359516,0.0,...,0.001068,0.998606,0.000302,0.000000,0.0,48962.455216,0.724802,35488.063349,0.998908,130207.5
